# OmniOpinion: Multi-Platform Social Media Ecosystem, Influencer Follower Dynamics & Dual-Valence Reactions (With vs. Without Traditional Press)

This notebook investigates:
1. **Most Accurate Brazilian Traditional Press Baseline ($M=3$ Tripolar Conglomerate Ecosystem):** Why the $M=3$ tripolar configuration ($\mathbf{m}_1 = +0.65, \tau_1 = 0.80$; $\mathbf{m}_2 = +0.15, \tau_2 = 0.25$; $\mathbf{m}_3 = -0.50, \tau_3 = 0.65$) most accurately represents the empirical Brazilian press reality (*Media Ownership Monitor Brasil*, Reuters Institute *Digital News Report*, Azevedo 2006/2017).
2. **Multi-Platform Social Media Usage by Brazilian Demographics:** Empirical reach across **Instagram/TikTok** (visual influencer & short-form video), **YouTube/X** (commentary & public microblogging), **Facebook** (social graph), and **WhatsApp/Telegram** (closed group cascades $K \le 256$).
3. **Influencer Network of Impact & Follower Gain/Loss Dynamics:** Grounded in *Preferential Attachment with Fitness* (Bianconi & Barabasi, 2001; Vosoughi et al., 2018) and *Dissonance-Driven Unfollowing* (Sasahara et al., 2021; Myers & Leskovec, 2014), comparing **Factual**, **Heterogeneous (Mixed)**, and **Tendentious** influencers.
4. **Dual-Valence Affective Reactions (Like vs. Dislike / Backfire):** How strong out-group dislike triggers moral outrage engagement and repulsive differentiation (Bail et al., 2018; Rathje et al., 2021; Brady et al., 2017).
5. **Simulations With vs. Without Traditional Press:** How institutional tripolar press ($M=3$) moderates or interacts with social media polarization and ideological dimensionality collapse.
6. **The `extreme_imposed` Scenario (1500 Steps / 1500-Run Supercomputer Design):** Deliberately choosing one extreme corner of the subject space ($\mathbf{e}_{\mathrm{imp}} = [+0.85, +0.85, +0.85, +0.85]$) where mostly tendentious influencers ($\tau_{\mathrm{inf}} \in [0.85, 0.98]$) have their posts and interactions artificially boosted in popularity and algorithmic reach — measuring both the population's tendency to follow `extreme_imposed` ideals and how `extreme_imposed` influencers become more popular than other influencers in general.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.18)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

sm_path = RESULTS_DIR / 'all_social_media_metrics.parquet'
df_sm = pd.read_parquet(sm_path)
print(f'Loaded social media dataset: {len(df_sm):,} rows across scenarios: {sorted(df_sm["scenario_name"].unique())}')

Loaded social media dataset: 28,836 rows across scenarios: ['sm_nopress_factual', 'sm_nopress_mixed', 'sm_nopress_tendentious', 'sm_withpress_m3_factual', 'sm_withpress_m3_mixed', 'sm_withpress_m3_tendentious']


In [2]:
# 1. Empirical Investigation of the Most Accurate Traditional Press Scenario before adding Social Media
df_trad = pd.read_parquet(RESULTS_DIR / 'all_trad_media_metrics.parquet')
df_tend = pd.read_parquet(RESULTS_DIR / 'all_tendentiousness_metrics.parquet')

trad_final = df_trad[(df_trad['city_scale'] == 'medium_100k') & (df_trad['step'] == df_trad['step'].max())]
summary_trad = trad_final.groupby('n_vehicle_groups')[[
    'esteban_ray', 'sarle_bimodality', 'dispersion_ratio_vs_random',
    'shift_toward_media_cm', 'mean_issue_correlation', 'effective_dimensionality', 'ideological_segregation'
]].mean().round(4)
print('=== Traditional Press Vehicle Groups (M = 0..4) Final Step Summary (N=100,000, Step=1500) ===')
print(summary_trad.to_string())

=== Traditional Press Vehicle Groups (M = 0..4) Final Step Summary (N=100,000, Step=1500) ===
                  esteban_ray  sarle_bimodality  dispersion_ratio_vs_random  shift_toward_media_cm  mean_issue_correlation  effective_dimensionality  ideological_segregation
n_vehicle_groups                                                                                                                                                             
0                      0.0059            0.4173                      0.3779                 0.0000                  0.0926                    3.6984                   0.1718
1                      0.1426            0.8177                      0.2263                 0.2933                  0.4169                    2.6236                   0.4647
2                      0.0782            0.1173                      0.1246                -0.0054                  0.3194                    3.0537                   0.3411
3                      0.0744       

In [3]:
# Figure SM1: Social Media With vs Without Traditional Press (M=3) across Factual, Mixed, and Tendentious Influencers
SCENARIO_LABELS = {
    'sm_nopress_factual': 'No Press (M=0) + Factual Influencers',
    'sm_nopress_mixed': 'No Press (M=0) + Mixed Influencers',
    'sm_nopress_tendentious': 'No Press (M=0) + Tendentious Influencers',
    'sm_withpress_m3_factual': 'With Tripolar Press (M=3) + Factual Influencers',
    'sm_withpress_m3_mixed': 'With Tripolar Press (M=3) + Mixed Influencers',
    'sm_withpress_m3_tendentious': 'With Tripolar Press (M=3) + Tendentious Influencers',
}
df_sm['scenario_label'] = df_sm['scenario_name'].map(SCENARIO_LABELS)

fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharex=True)
palette = {
    'No Press (M=0) + Factual Influencers': '#2ca02c',
    'No Press (M=0) + Mixed Influencers': '#ff7f0e',
    'No Press (M=0) + Tendentious Influencers': '#d62728',
    'With Tripolar Press (M=3) + Factual Influencers': '#1f77b4',
    'With Tripolar Press (M=3) + Mixed Influencers': '#9467bd',
    'With Tripolar Press (M=3) + Tendentious Influencers': '#8c564b',
}

metrics_grid = [
    ('esteban_ray', r'Esteban-Ray Polarization ($P_{ER}$)', axes[0, 0]),
    ('dispersion_ratio_vs_random', r'Dispersion Ratio vs. Uniform Random ($\mathrm{Var}/(1/3)$)', axes[0, 1]),
    ('mean_issue_correlation', r'Cross-Issue Alignment ($\bar{|\rho_{ab}|}$)', axes[1, 0]),
    ('effective_dimensionality', r'Effective Attitude Dimensionality ($\mathrm{PR}(\Sigma)$)', axes[1, 1]),
]

for col, title, ax in metrics_grid:
    sns.lineplot(
        data=df_sm,
        x='step',
        y=col,
        hue='scenario_label',
        style='press_mode',
        palette=palette,
        linewidth=2.2,
        ax=ax,
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Simulation Step ($t$)')
    ax.legend(fontsize=8.5, loc='best')

plt.suptitle(
    'Social Media Layer With vs. Without Brazilian Tripolar Press (M=3): Polarization & Dimensionality',
    fontsize=14,
    fontweight='bold',
    y=0.995,
)
plt.tight_layout()
out_fig1 = FIGURES_DIR / 'fig_sm1_with_vs_without_press_trajectories.png'
plt.savefig(out_fig1, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig1}')

<Figure size 2100x1540 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_sm1_with_vs_without_press_trajectories.png


In [4]:
# Figure SM2: Platform Usage Reach, Dual-Valence Reactions (Like vs Dislike), and Influencer Follower Dynamics
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Panel A: Empirical Brazilian Demographic Reach curves by Age (SES = 0.40 median)
ages = np.linspace(16, 90, 200)
age_norm = (ages - 16.0) / 74.0
ses_med = 0.40
p_ig_tt = np.clip(0.84 - 0.62 * (age_norm ** 0.85) + 0.10 * ses_med, 0.08, 0.94)
p_yt_x = np.clip(0.56 - 0.28 * age_norm + 0.24 * (ses_med ** 1.2), 0.12, 0.88)
p_fb = np.clip(0.22 + 0.44 * np.sin(np.pi * (age_norm ** 0.75)) - 0.08 * ses_med, 0.10, 0.78)
p_tv = np.clip(0.24 + 0.48 * (age_norm ** 0.9) - 0.10 * ses_med, 0.05, 0.85)

ax = axes[0, 0]
ax.plot(ages, p_ig_tt * 100, label='Instagram / TikTok (Visual Influencer)', color='#e377c2', lw=2.5)
ax.plot(ages, p_yt_x * 100, label='YouTube / X (Video & Microblog)', color='#d62728', lw=2.5)
ax.plot(ages, p_fb * 100, label='Facebook (Social Graph Feed)', color='#1f77b4', lw=2.5)
ax.plot(ages, p_tv * 100, label='Broadcast TV (Traditional Reference)', color='#5c6b73', lw=2.2, ls='--')
ax.set_title('A. Calibrated Brazilian Media Reach by Age Cohort (IBGE / DataReportal 2024)', fontweight='bold')
ax.set_xlabel('Agent Age (Years)')
ax.set_ylabel('Step Access Probability (%)')
ax.legend(fontsize=9)

# Panel B: Dual-Valence Dislike Share across Influencer Tendentiousness & Press Regimes
ax = axes[0, 1]
sns.lineplot(
    data=df_sm[df_sm['step'] > 0],
    x='step',
    y='dislike_share',
    hue='scenario_label',
    palette=palette,
    linewidth=2.0,
    ax=ax,
)
ax.set_title('B. Out-Group Dislike / Moral Outrage Share of Total Social Reactions', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Dislike Reaction Share ($N_{\\mathrm{dislike}} / N_{\\mathrm{total}}$)')
ax.legend(fontsize=8)

# Panel C: Influencer Follower Inequality (Gini Coefficient) Trajectory
ax = axes[1, 0]
sns.lineplot(
    data=df_sm[df_sm['step'] > 0],
    x='step',
    y='influencer_follower_gini',
    hue='scenario_label',
    palette=palette,
    linewidth=2.2,
    ax=ax,
)
ax.set_title('C. Influencer Audience Concentration (Follower Gini Coefficient)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Follower Gini Index')
ax.legend(fontsize=8)

# Panel D: Mean Influencer Followers Growth Trajectory
ax = axes[1, 1]
sns.lineplot(
    data=df_sm[df_sm['step'] > 0],
    x='step',
    y='mean_inf_followers',
    hue='scenario_label',
    palette=palette,
    linewidth=2.2,
    ax=ax,
)
ax.set_title('D. Influencer Network of Impact: Mean Follower Count Evolution', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Followers per Influencer')
ax.legend(fontsize=8)

plt.tight_layout()
out_fig2 = FIGURES_DIR / 'fig_sm2_platforms_dislike_and_influencer_dynamics.png'
plt.savefig(out_fig2, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig2}')

<Figure size 2100x1540 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_sm2_platforms_dislike_and_influencer_dynamics.png


In [5]:
# Final Comparative Summary Table: With vs. Without Traditional Press across Influencer Regimes
final_sm = df_sm[df_sm['step'] == df_sm['step'].max()]
cols = [
    'esteban_ray', 'sarle_bimodality', 'dispersion_ratio_vs_random',
    'mean_issue_correlation', 'effective_dimensionality', 'ideological_segregation',
    'dislike_share', 'mean_inf_followers', 'influencer_follower_gini', 'whatsapp_cascades'
]
table_sm = final_sm.groupby(['press_mode', 'influencer_mode'])[cols].mean().round(4)
print('=== Final Equilibrium Comparison: Social Media With vs. Without Traditional Press ===')
print(table_sm.to_string())

=== Final Equilibrium Comparison: Social Media With vs. Without Traditional Press ===
                                                 esteban_ray  sarle_bimodality  dispersion_ratio_vs_random  mean_issue_correlation  effective_dimensionality  ideological_segregation  dislike_share  mean_inf_followers  influencer_follower_gini  whatsapp_cascades
press_mode                      influencer_mode                                                                                                                                                                                                                      
With Traditional Press (M=3)    factual               0.0642            0.3719                      0.1392                  0.1108                    3.5188                   0.4331         0.0119           1267.6376                    0.4895         28009.0000
                                heterogeneous         0.0772            0.4938                      0.1834                  0.20

## 6. The `extreme_imposed` Scenario: Population Alignment & Influencer Dominance

In this scenario (`extreme_imposed`), one extreme corner of the $4$-dimensional subject space ($\mathbf{e}_{\mathrm{imp}} = [+0.85, +0.85, +0.85, +0.85]$) is deliberately chosen:
- **Mostly Tendentious Influencers ($\tau_{\mathrm{inf}} \in [0.85, 0.98]$):** $25\%$ of the influencer pool is anchored in the `extreme_imposed` region.
- **Artificial Popularity & Reach Amplification:** Their posts and interactions receive an artificial popularity multiplier ($\beta_{\mathrm{pop}} \in \{1.0, 4.0, 8.0\}$) and algorithmic reach multiplier ($\beta_{\mathrm{reach}} \in \{1.0, 3.5, 7.0\}$), grounded in **Huszár et al. (2022, *PNAS*)** and **Muchnik, Aral, & Taylor (2013, *Science*)**.
- **Key Measures Across 1500 Steps:**
  1. **Population Tendency to Follow `extreme_imposed` Ideals:** `frac_following_extreme_imposed` (share of citizens within normalized Euclidean distance $\le 0.45$ of $\mathbf{e}_{\mathrm{imp}}$) and `extreme_imposed_alignment` (projection of population center of mass onto $\hat{\mathbf{e}}_{\mathrm{imp}}$).
  2. **Dominance of `extreme_imposed` Influencers Over Other Influencers:** `extreme_imposed_follower_ratio` ($\bar{F}_{\mathrm{ext}} / \bar{F}_{\mathrm{other}}$), `extreme_imposed_follower_share`, and `extreme_imposed_popularity_ratio` ($\bar{P}_{\mathrm{ext}} / \bar{P}_{\mathrm{other}}$).

In [6]:
ext_path = RESULTS_DIR / 'all_extreme_imposed_metrics.parquet'
df_ext = pd.read_parquet(ext_path)

ext_label_map = {
    'ext_nopress_baseline': 'No Press (M=0) | Baseline (1x)',
    'ext_nopress_moderate_boost': 'No Press (M=0) | Moderate Boost (3.5x Reach / 4x Pop)',
    'ext_nopress_high_boost': 'No Press (M=0) | High Boost (7x Reach / 8x Pop)',
    'ext_withpress_m3_baseline': 'With Press (M=3) | Baseline (1x)',
    'ext_withpress_m3_moderate_boost': 'With Press (M=3) | Moderate Boost (3.5x Reach / 4x Pop)',
    'ext_withpress_m3_high_boost': 'With Press (M=3) | High Boost (7x Reach / 8x Pop)',
}
df_ext['scenario_label'] = df_ext['scenario_name'].map(ext_label_map)

ext_final = df_ext[df_ext['step'] == df_ext['step'].max()]
ext_cols = [
    'frac_following_extreme_imposed',
    'extreme_imposed_alignment',
    'dist_to_extreme_imposed_mean',
    'extreme_imposed_inf_followers_mean',
    'other_inf_followers_mean',
    'extreme_imposed_follower_ratio',
    'extreme_imposed_follower_share',
    'extreme_imposed_popularity_ratio',
    'extreme_imposed_popularity_share',
]
ext_summary = ext_final.groupby(['press_mode', 'boost_level'])[ext_cols].mean().round(4)
print('=== Extreme Imposed Scenario Summary at Step 1500 ===')
print(ext_summary.to_string())

=== Extreme Imposed Scenario Summary at Step 1500 ===
                                                          frac_following_extreme_imposed  extreme_imposed_alignment  dist_to_extreme_imposed_mean  extreme_imposed_inf_followers_mean  other_inf_followers_mean  extreme_imposed_follower_ratio  extreme_imposed_follower_share  extreme_imposed_popularity_ratio  extreme_imposed_popularity_share
press_mode                      boost_level                                                                                                                                                                                                                                                                                                        
With Traditional Press (M=3)    Baseline (1x)                                     0.0104                     0.0633                        0.8375                           1227.8190                 1244.6399                          0.9874                          0

In [7]:
# Figure SM3: Extreme Imposed Scenario — Population Alignment & Influencer Dominance (1500 Steps)
fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

ext_palette = {
    'No Press (M=0) | Baseline (1x)': '#7f7f7f',
    'No Press (M=0) | Moderate Boost (3.5x Reach / 4x Pop)': '#ff7f0e',
    'No Press (M=0) | High Boost (7x Reach / 8x Pop)': '#d62728',
    'With Press (M=3) | Baseline (1x)': '#aec7e8',
    'With Press (M=3) | Moderate Boost (3.5x Reach / 4x Pop)': '#1f77b4',
    'With Press (M=3) | High Boost (7x Reach / 8x Pop)': '#9467bd',
}

# Panel A: Population Fraction Following Extreme Imposed Ideals
ax = axes[0, 0]
sns.lineplot(
    data=df_ext,
    x='step',
    y='frac_following_extreme_imposed',
    hue='scenario_label',
    style='press_mode',
    palette=ext_palette,
    linewidth=2.3,
    ax=ax,
)
ax.set_title(r'A. Population Fraction Following Extreme Imposed Ideals ($d(\mathbf{x}_i, \mathbf{e}_{\mathrm{imp}}) \leq 0.45$)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Fraction of Population Following Ideals')
ax.legend(fontsize=8, loc='upper left')

# Panel B: Population Center-of-Mass Alignment toward Extreme Imposed Corner
ax = axes[0, 1]
sns.lineplot(
    data=df_ext,
    x='step',
    y='extreme_imposed_alignment',
    hue='scenario_label',
    style='press_mode',
    palette=ext_palette,
    linewidth=2.3,
    ax=ax,
)
ax.set_title(r'B. Population Ideological Alignment with $\mathbf{e}_{\mathrm{imp}}$ ($\bar{\mathbf{x}} \cdot \hat{\mathbf{e}}_{\mathrm{imp}}$)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel(r'Directional Alignment ($[-1, +1]$)')
ax.legend(fontsize=8, loc='upper left')

# Panel C: Influencer Follower Ratio (Extreme Imposed vs. Other Influencers)
ax = axes[1, 0]
sns.lineplot(
    data=df_ext[df_ext['step'] > 0],
    x='step',
    y='extreme_imposed_follower_ratio',
    hue='scenario_label',
    style='press_mode',
    palette=ext_palette,
    linewidth=2.3,
    ax=ax,
)
ax.axhline(1.0, color='black', ls=':', lw=1.3, label='Parity (1.0x)')
ax.set_title(r'C. Influencer Follower Dominance Ratio ($\bar{F}_{\mathrm{ext}} / \bar{F}_{\mathrm{other}}$)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Follower Ratio vs. Other Influencers')
ax.legend(fontsize=8, loc='upper left')

# Panel D: Influencer Post Popularity Share vs. Creator Share (25%)
ax = axes[1, 1]
sns.lineplot(
    data=df_ext[df_ext['step'] > 0],
    x='step',
    y='extreme_imposed_popularity_share',
    hue='scenario_label',
    style='press_mode',
    palette=ext_palette,
    linewidth=2.3,
    ax=ax,
)
ax.axhline(0.25, color='black', ls=':', lw=1.3, label='Baseline Creator Share (25%)')
ax.set_title('D. Share of Total Platform Post Popularity Captured by Extreme Imposed Influencers', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Share of Total Ecosystem Popularity')
ax.legend(fontsize=8, loc='lower right')

plt.suptitle(
    'Extreme Imposed Scenario (1500 Steps): Population Ideological Shift & Tendentious Influencer Dominance',
    fontsize=14,
    fontweight='bold',
    y=0.995,
)
plt.tight_layout()
out_fig3 = FIGURES_DIR / 'fig_sm3_extreme_imposed_population_and_influencers.png'
plt.savefig(out_fig3, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig3}')

<Figure size 2240x1610 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_sm3_extreme_imposed_population_and_influencers.png


## 7. Long-Term Synchronization Tactics, Influencer Co-Working & Intermedia Dictation of Subjects and Facts

This section evaluates the long-term impact (`1500` steps) when the **`extreme_imposed`** influencer group combines three literature-grounded coordination mechanisms:
1. **Tendentious & Synchronized Focal-Fact Campaigns (`sync_tactics_enabled`, Pacheco et al., 2021, *ICWSM*; Giglietto et al., 2020; Pennycook et al., 2018):** Instead of posting dispersed narratives, `extreme_imposed` influencers synchronize their posts around a specific focal fact $k^*$ with tendentious framing toward $\mathbf{e}_{\mathrm{imp}}$, generating illusory-truth multi-source reinforcement.
2. **Influencer Co-Working & Cross-Network Access (`cowork_enabled`, Lewis, 2018, *Alternative Influence*; Ribeiro et al., 2020, *ACM FAT*):** Influencers collaborate with other influencers, directly accessing their partner's follower networks (`cowork_cross_exposures`) with co-host parasocial endorsement and cross-recruiting followers to expand their network carrying capacity.
3. **Intermedia Dictation of Subjects & Facts to Traditional Media (`fact_dictation_enabled`, McCombs, 2004; Harder et al., 2017; Gilardi et al., 2022, *Political Communication*; Downs, 1972):** Facts noticed and amplified by the synchronized influencer network receive a surge in public interest (`sync_network_interest_push`) that forces traditional media ($M=3$: TV, Radio, Written Press) to notice and cover those facts (`dictated_trad_share`), pulls traditional framing toward the dictated subject angle, and extends fact longevity (`mean_fact_longevity`) so high-relevance pushed facts persist much longer.

In [8]:
sync_path = RESULTS_DIR / 'all_sync_dictation_metrics.parquet'
df_sync = pd.read_parquet(sync_path)

sync_label_map = {
    'unsync_nopress': '1. No Press (M=0) | Unsynchronized Extreme Imposed',
    'sync_cowork_nopress': '2. No Press (M=0) | Synchronized + Co-Work',
    'unsync_withpress_m3': '3. With Press (M=3) | Unsynchronized Extreme Imposed',
    'sync_only_withpress': '4. With Press (M=3) | Synchronized Facts Only',
    'sync_cowork_withpress': '5. With Press (M=3) | Synchronized + Co-Work',
    'full_sync_cowork_dict': '6. With Press (M=3) | Full Sync + Co-Work + Fact Dictation',
}
df_sync['scenario_label'] = df_sync['scenario_name'].map(sync_label_map)

sync_final = df_sync[df_sync['step'] == df_sync['step'].max()]
sync_cols = [
    'frac_following_extreme_imposed',
    'extreme_imposed_alignment',
    'extreme_imposed_inf_followers_mean',
    'other_inf_followers_mean',
    'extreme_imposed_follower_ratio',
    'extreme_imposed_follower_share',
    'cowork_cross_exposures',
    'dictated_trad_share',
    'mean_fact_longevity',
]
sync_summary = sync_final.groupby('scenario_label')[sync_cols].mean().round(4)
print('=== Long-Term Synchronization, Co-Working & Fact Dictation Summary (Step 1500) ===')
print(sync_summary.to_string())

=== Long-Term Synchronization, Co-Working & Fact Dictation Summary (Step 1500) ===
                                                            frac_following_extreme_imposed  extreme_imposed_alignment  extreme_imposed_inf_followers_mean  other_inf_followers_mean  extreme_imposed_follower_ratio  extreme_imposed_follower_share  cowork_cross_exposures  dictated_trad_share  mean_fact_longevity
scenario_label                                                                                                                                                                                                                                                                                                       
1. No Press (M=0) | Unsynchronized Extreme Imposed                                  0.7148                     0.5720                           4091.2572                  678.8665                          6.0693                          0.6683                     0.0               0.0000         

In [9]:
# Figure SM4: Long-Term Synchronization Tactics, Influencer Co-Working & Intermedia Fact Dictation
fig, axes = plt.subplots(2, 2, figsize=(16.5, 11.5))

sync_palette = {
    '1. No Press (M=0) | Unsynchronized Extreme Imposed': '#ff9896',
    '2. No Press (M=0) | Synchronized + Co-Work': '#d62728',
    '3. With Press (M=3) | Unsynchronized Extreme Imposed': '#aec7e8',
    '4. With Press (M=3) | Synchronized Facts Only': '#1f77b4',
    '5. With Press (M=3) | Synchronized + Co-Work': '#2ca02c',
    '6. With Press (M=3) | Full Sync + Co-Work + Fact Dictation': '#9467bd',
}

# Panel A: Population Fraction Following Extreme Imposed Ideals
ax = axes[0, 0]
sns.lineplot(
    data=df_sync,
    x='step',
    y='frac_following_extreme_imposed',
    hue='scenario_label',
    palette=sync_palette,
    linewidth=2.3,
    ax=ax,
)
ax.set_title(r'A. Long-Term Population Capture by Extreme Imposed Ideals ($d(\mathbf{x}_i, \mathbf{e}_{\mathrm{imp}}) \leq 0.45$)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Fraction of Population Following Ideals')
ax.legend(fontsize=7.8, loc='lower right')

# Panel B: Mean Followers of Extreme Imposed Influencers (Co-Working Network Expansion)
ax = axes[0, 1]
sns.lineplot(
    data=df_sync[df_sync['step'] > 0],
    x='step',
    y='extreme_imposed_inf_followers_mean',
    hue='scenario_label',
    palette=sync_palette,
    linewidth=2.3,
    ax=ax,
)
ax.set_title('B. Influencer Co-Working Network Growth: Mean Followers per Extreme Imposed Influencer', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Followers (Extreme Imposed Creators)')
ax.legend(fontsize=7.8, loc='lower right')

# Panel C: Traditional Media Agenda Dictation Share (Share of Active Traditional Coverage Driven by Network Facts)
ax = axes[1, 0]
sns.lineplot(
    data=df_sync[df_sync['step'] > 0],
    x='step',
    y='dictated_trad_share',
    hue='scenario_label',
    palette=sync_palette,
    linewidth=2.3,
    ax=ax,
)
ax.set_title('C. Intermedia Fact Dictation: Share of Traditional Media Coverage Driven by Pushed Facts', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Dictated Share of Traditional Media Intensity')
ax.legend(fontsize=7.8, loc='upper left')

# Panel D: Active Fact Longevity Multiplier (Persistence Extension of High-Interest Facts)
ax = axes[1, 1]
sns.lineplot(
    data=df_sync[df_sync['step'] > 0],
    x='step',
    y='mean_fact_longevity',
    hue='scenario_label',
    palette=sync_palette,
    linewidth=2.0,
    ax=ax,
)
ax.axhline(1.0, color='black', ls=':', lw=1.3, label='Baseline Fact Decay (1.0x)')
ax.set_title('D. Fact Relevance & Longevity Multiplier (Extended News-Cycle Persistence)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Fact Longevity Multiplier ($L_k$)')
ax.legend(fontsize=7.8, loc='upper right')

plt.suptitle(
    'Long-Term Impact of Synchronization Tactics, Influencer Co-Working & Intermedia Fact Dictation (1500 Steps)',
    fontsize=14,
    fontweight='bold',
    y=0.995,
)
plt.tight_layout()
out_fig4 = FIGURES_DIR / 'fig_sm4_sync_cowork_and_fact_dictation.png'
plt.savefig(out_fig4, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig4}')

<Figure size 2310x1610 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_sm4_sync_cowork_and_fact_dictation.png
